# 5. AI와 함께 분석하기

웹 강의자료: [https://statjeong.github.io/moj-python-lecture/05_ai.html](https://statjeong.github.io/moj-python-lecture/05_ai.html)

ChatGPT, Claude 같은 AI는 분석 코드를 빠르게 써 줍니다. 이 장에서는 AI에게 원하는 코드를 받아내는 요청 방법과, 받은 코드가 어떻게 작동하는지 묻고 결과를 검증하는 방법을 연습합니다. AI가 데이터 분석 코드를 쓰고 실행할 때 주로 쓰는 언어가 파이썬이라서, 1~4장에서 익힌 문법이 AI가 준 코드를 읽는 데 그대로 쓰입니다.

## 5.1 준비

In [ ]:
import pandas as pd

주소 = "https://raw.githubusercontent.com/StatJeong/moj-python-lecture/main/data/"
수용자 = pd.read_csv(주소 + "inmates.csv")
시설현황 = pd.read_csv(주소 + "facilities.csv")

## 5.2 AI에게 일을 맡기는 순서

1. 데이터 구조를 알려준다 (열 이름, 자료형, 가상의 예시 몇 줄)
2. 원하는 결과를 구체적으로 적는다
3. 조건을 적는다 (실행 환경, 사용할 도구, 주석)
4. 받은 코드를 실행한다
5. 코드가 무엇을 하는지 묻고, 결과가 맞는지 확인한다

## 5.3 데이터 구조를 알려주는 방법

AI에게는 데이터 전체가 아니라 구조만 알려주면 충분합니다.

In [ ]:
print(수용자.dtypes)

In [ ]:
print(수용자.head(3).to_string())

실제 업무 데이터라면 위처럼 값을 보여주지 말고, 열 이름과 자료형만 알려주거나 값을 가상으로 바꾼 예시를 만들어 알려줍니다. 수용자 이름, 수용번호, 상담 내용 같은 실제 정보는 외부 AI 서비스에 입력하지 않습니다.

## 5.4 요청문 틀

아래 틀을 복사해 내용만 바꿔 씁니다.

```text
[상황]
파이썬 pandas로 수용자 데이터를 분석하고 있습니다.
데이터프레임 이름은 수용자입니다.

[데이터 구조]
(print(수용자.dtypes) 결과를 붙여 넣기)

[원하는 결과]
경비처우급별 평균 징벌횟수를 구하고 막대그래프로 그려주세요.

[조건]
- Google Colab에서 실행합니다.
- 한글 글꼴(NanumGothic)은 이미 설치되어 있습니다.
- 각 줄에 한국어 주석을 달아주세요.
```

## 5.5 받은 코드에 질문하기

코드를 받으면 다음 세 단계를 거칩니다.

1. 실행해 본다. 오류가 나면 오류 메시지를 그대로 AI에게 붙여 넣고 묻는다.
2. 한 줄씩 무엇을 하는지 묻는다.
3. 결과가 맞는지 확인하는 방법을 묻고, 직접 확인한다.

질문 예시입니다.

- "3번째 줄의 groupby는 어떤 일을 하나요?"
- "이 코드에서 비어 있는 값(결측치)은 어떻게 처리되나요?"
- "결과가 맞는지 손으로 확인하려면 어떻게 하면 되나요?"
- "같은 결과를 다른 방법으로도 낼 수 있나요?"

예를 들어 위 요청문에 AI가 다음 코드를 줬다고 합시다.

In [ ]:
결과 = 수용자.groupby("경비처우급")["징벌횟수"].mean()   # 경비처우급별 징벌횟수 평균
결과

결과 중 하나를 직접 계산해 맞는지 확인합니다. S4인 수용자만 골라 평균을 내 봅니다.

In [ ]:
S4 = 수용자[수용자["경비처우급"] == "S4"]
print(S4[["수용번호", "징벌횟수"]])
print(S4["징벌횟수"].mean())

## 5.6 AI가 준 코드에서 자주 생기는 문제

아래 예들은 모두 오류 없이 실행됩니다. 그래서 더 놓치기 쉽습니다.

### 결측치가 조용히 빠지는 평균

In [ ]:
print(수용자["상담횟수"].mean())                    # 값이 있는 9명으로 나눔
print(수용자["상담횟수"].sum() / len(수용자))       # 10명으로 나눔

"평균 상담횟수"를 요청하면 AI는 보통 첫 번째 방법을 씁니다. 어느 쪽이 맞는지는 빈칸의 의미에 따라 달라지므로, 요청할 때 결측치 처리 방법을 적거나 받은 뒤에 물어봐야 합니다.

### 비율의 평균

In [ ]:
시설현황["수용률"] = 시설현황["현원"] / 시설현황["정원"] * 100
print(시설현황["수용률"].mean())                                # 시설별 수용률의 단순 평균
print(시설현황["현원"].sum() / 시설현황["정원"].sum() * 100)    # 전체 현원 / 전체 정원

"전체 수용률"을 원했다면 두 번째가 맞습니다. 시설마다 규모가 다른데 첫 번째 방법은 시설 4곳을 같은 무게로 평균 냈기 때문에 값이 다릅니다.

### 개월을 30일로 바꾸는 날짜 계산

In [ ]:
수용자["입소일"] = pd.to_datetime(수용자["입소일"])

# 형기 1개월을 30일로 바꿔 더하는 방법
수용자["출소_30일근사"] = 수용자["입소일"] + pd.to_timedelta(수용자["형기_개월"] * 30, unit="D")

# 달력 기준으로 개월을 더하는 방법
달력기준 = []
for 입소일, 형기 in zip(수용자["입소일"], 수용자["형기_개월"]):
    달력기준.append(입소일 + pd.DateOffset(months=int(형기)))
수용자["출소_달력기준"] = 달력기준

수용자["차이_일"] = (수용자["출소_달력기준"] - 수용자["출소_30일근사"]).dt.days
수용자[["수용번호", "형기_개월", "출소_30일근사", "출소_달력기준", "차이_일"]]

형기가 길수록 차이가 커져서 60개월인 1008번은 26일이 차이 납니다. 날짜 계산 코드를 받으면 "개월은 어떻게 더했나요?"라고 확인합니다.

### 오류 없이 끝나지만 믿으면 안 되는 회귀 결과

4.7절에서 본 완전 분리처럼, 경고 메시지가 나오거나 계수와 표준오차가 비정상적으로 크면 결과를 그대로 쓰지 않습니다. AI에게 "이 경고는 무슨 뜻인가요?"라고 묻습니다.

## 5.7 실습 과제

아래 과제를 AI에게 요청해 해결해 보세요. 받은 코드는 5.5절의 세 단계로 확인합니다. 정답 보기에는 확인용 코드가 있습니다.

1. 기준일 2026년 12월 1일로부터 6개월 이내(2027년 6월 1일 이전)에 출소 예정인 수용자를 찾으세요.

In [ ]:
# AI에게 받은 코드를 붙여 넣어 실행해 보세요

2. 시설별로 인원, 평균 형기(개월), 징벌을 한 번 이상 받은 사람의 비율을 한 표로 만드세요.

In [ ]:
# AI에게 받은 코드를 붙여 넣어 실행해 보세요

3. 최근상담메모를 "생활", "심리", "진로" 중 하나로 분류하게 해 보세요. 아래 코드로 메모를 출력한 뒤, 그 내용을 AI에게 주고 분류를 요청합니다.

In [ ]:
for 번호, 메모 in zip(수용자["수용번호"], 수용자["최근상담메모"]):
    print(번호, 메모)

이 메모는 강의용 가상 문장입니다. 실제 상담기록은 외부 AI에 입력하지 않습니다.

4. 3.5절의 시설별 수용률 표를 AI에게 주고 세 문장짜리 보고서 문단을 써 달라고 요청하세요. 받은 문장에 나온 숫자가 표의 숫자와 같은지 하나씩 확인합니다.

## 5.8 실제 공개 데이터로 해보기

(준비 중) 교정통계연보 등 공개 통계를 내려받아 같은 순서로 분석합니다.

## 5.9 AI를 쓸 때 지킬 것

- 실제 수용자 정보, 상담 내용 등 민감정보는 외부 AI 서비스에 입력하지 않습니다.
- 열 이름, 자료형, 가상 예시만 알려주고 받은 코드는 내부 환경에서 실행합니다.
- 결과 숫자는 일부라도 직접 계산해 확인합니다.
- 경고 메시지와 비정상적인 값은 그냥 넘기지 않고 묻습니다.
- 기관의 정보보안 지침을 따릅니다.